In [1]:
# =========================
# Imports
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [17]:
# =========================
# Load Data
# =========================
df = pd.read_csv("/Users/ameypatil/Desktop/Tata Power/data/2024 data/autumn_2024.csv")  # change path

# Ensure sorted

# Datetime handling
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')

# Feature
df['hour'] = df['datetime'].dt.hour

# Drop NaN
df = df.dropna()

df.head()

,datetime,wdLoad,temp,rh,year,month,hour
0,2024-10-01 00:00:00+05:30,347.735464,28.566667,92.900000,2024,10,0
1,2024-10-01 00:15:00+05:30,342.400444,28.500000,92.833333,2024,10,0
2,2024-10-01 00:30:00+05:30,336.369117,28.400000,93.266667,2024,10,0
3,2024-10-01 00:45:00+05:30,331.087170,28.300000,93.600000,2024,10,0
4,2024-10-01 01:00:00+05:30,326.584807,28.233333,93.933333,2024,10,1


In [18]:
# =========================
# Define Variables
# =========================

# Outcome
Y = df['wdLoad'].values

# Multi-treatment (temp + humidity)
T = df[['temp', 'rh']].values

# Confounders
X = df[['hour']].values  # add more if needed

In [19]:
# =========================
# Cross-Fitting DML
# =========================

kf = KFold(n_splits=10, shuffle=True, random_state=42)

Y_res = np.zeros_like(Y)
T_res = np.zeros_like(T)

for train_idx, test_idx in kf.split(X):
    
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]
    
    # Models
    model_y = RandomForestRegressor(n_estimators=100, random_state=42)
    model_t = RandomForestRegressor(n_estimators=100, random_state=42)
    
    # Fit
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    # Residuals
    Y_res[test_idx] = Y_test - model_y.predict(X_test)
    T_res[test_idx] = T_test - model_t.predict(X_test)

# Final stage
final_model = LinearRegression()
final_model.fit(T_res, Y_res)

beta_temp = final_model.coef_[0]
beta_rh = final_model.coef_[1]

print("ATE (Temp):", beta_temp)
print("ATE (RH):", beta_rh)

ATE (Temp): 4.4945227411215765
ATE (RH): 1.0994025128063583
